# 46. 질문 조건 추출과 실패 경계

목적: 빈 AND, 추출 실패, 명시 카드 범위 손실을 먼저 재현하고 실험용 경계 정책을 확인합니다. 운영 코드와 과거 결과를 수정하지 않습니다.

개발 자료: 통제 반례와 정상·변형 12쌍을 각각 3회 추출합니다. 원문에 없는 수치나 조건을 보정으로 추가하지 않습니다. API 전체 예산은 46~49 합계 8달러입니다. 응답시간은 측정하며 운영 SLA는 아직 정하지 않았습니다.

공유 보조 코드가 필요한 이유: 46~49에서 같은 실패 경계와 누적 비용 차단을 사용해야 합니다. 이를 한 파일로 재사용하고 각 실험의 실행·평가·중단 재개는 노트북에서 수행합니다. 새 보고서 생성기는 만들지 않습니다.

In [1]:
from pathlib import Path
import importlib.util, sys, json, time, resource, hashlib, os
ROOT = Path.cwd()
if not (ROOT / 'notebooks').exists():
    ROOT = ROOT.parent
assert (ROOT / 'notebooks/scripts').is_dir()
sys.dont_write_bytecode = True
def load(name, filename):
    spec = importlib.util.spec_from_file_location(name, ROOT / 'notebooks/scripts' / filename)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod
OLD = load('boundary46_old', '43_routing_replay.py')
FIX = json.loads((ROOT/'notebooks/data/43_offline_validation/routing_fixtures.json').read_text())
PATH = ROOT/'notebooks/data/46_query_boundary_validation'
PATH.mkdir(exist_ok=True)
F = load('boundary46_io', '34_followup_common.py')
F.STAGES[46] = PATH
F.initialize(46)
def write(name, value): F.write_json(PATH/name, value)
def req(target='커피', **kw):
    return dict(targets=[target], action='discount', min_percent=10, max_spend_krw=None,
                min_cap_krw=None, cap_period='any', exclude_targets=[], **kw)
def plan(groups=None, cards=None):
    return dict(route='structured_then_unstructured', groups=[[req()]] if groups is None else groups,
                card_keys=cards or [], unresolved=[])
# 기대값은 선택기나 새 정책으로 계산하지 않은 리터럴입니다.
REGRESSIONS = [
    dict(id='empty_inner_and', value=plan([[]]), explicit=None, failed=False, unclear=False,
         want={'route':'unstructured','scope':None}),
    dict(id='failed_parser_preserves_A', value=plan(cards=['A']), explicit=['A'], failed=True, unclear=False,
         want={'route':'unstructured','scope':['A']}),
    dict(id='failed_parser_must_not_trust_B', value=plan(cards=['B']), explicit=None, failed=True, unclear=False,
         want={'route':'unstructured','scope':None}),
    dict(id='unknown_card_preserves_A', value=plan(cards=['Z']), explicit=['A'], failed=False, unclear=False,
         want={'route':'unstructured','scope':['A']}),
    dict(id='unclear_scope_requires_clarification', value=plan(), explicit=None, failed=False, unclear=True,
         want={'route':'clarify','scope':[]}),
    dict(id='empty_outer_group', value=plan([]), explicit=None, failed=False, unclear=False,
         want={'route':'unstructured','scope':None}),
    dict(id='empty_requirement', value=plan([[dict(targets=[],action='any',min_percent=None,
         max_spend_krw=None,min_cap_krw=None,cap_period='any',exclude_targets=[])]]), explicit=None,
         failed=False, unclear=False, want={'route':'unstructured','scope':None}),
    dict(id='same_card_AND', value=plan([[req(),req('넷플릭스')]]), explicit=None, failed=False, unclear=False,
         want={'route':'structured_then_unstructured','scope':['A']}),
    dict(id='no_match_preserves_B', value=plan([[req('병원')]],['B']), explicit=['B'], failed=False, unclear=False,
         want={'route':'unstructured','scope':['B']}),
]
def run_regressions(fn):
    rows=[]
    for c in REGRESSIONS:
        got=fn(c)
        rows.append(dict(id=c['id'], want=c['want'], got={k:got[k] for k in c['want']},
                         passed=all(got[k]==v for k,v in c['want'].items())))
    return rows
print('준비:',len(REGRESSIONS),'경계 회귀')

준비: 9 경계 회귀


In [2]:
def old_adapter(c):
    row=OLD.replay(c['value'],FIX['facts'],FIX['catalog'],parser_failed=c['failed'])
    return dict(route='unstructured' if row['fallback'] else 'structured_then_unstructured',
                scope=row['effective_scope'])
red=run_regressions(old_adapter)
write('red_regression.json', {'records':red,'expected_failure':True,'new_api_requests':0})
assert any(not r['passed'] for r in red), '기존 문제를 재현하지 못했습니다.'
print('기존 동작에서 재현된 실패:', [r['id'] for r in red if not r['passed']])

기존 동작에서 재현된 실패: ['empty_inner_and', 'failed_parser_preserves_A', 'unknown_card_preserves_A', 'unclear_scope_requires_clarification', 'empty_requirement']


In [3]:
P = load('boundary46_policy', '46_preproduction_common.py')
def new_adapter(c):
    return P.route('',c['value'],FIX['facts'],FIX['catalog'],trusted_scope=c['explicit'],
                   parser_failed=c['failed'],scope_unclear=c['unclear'])
green=run_regressions(new_adapter)
write('green_regression.json', {'records':green,'new_api_requests':0})
assert all(r['passed'] for r in green), green
# 43번의 나머지 사례도 현재 입력 계약으로 재생합니다. 옛 예상값은 역사 기록입니다.
semantic_questions={
 'semantic_target':'커피를 20% 이상 할인해주는 카드',
 'semantic_action':'커피를 10% 이상 할인해주는 카드',
 'semantic_number':'커피를 20% 이상 할인해주는 카드',
 'semantic_boolean':'커피와 넷플릭스를 각각 10% 이상 할인해주는 카드 한 장',
 'semantic_period':'커피를 5% 이상 할인하고 월 할인한도가 5000원 이상인 카드',
 'semantic_card_scope':'A 카드의 커피 10% 할인',
}
controlled=[]
for c in FIX['cases']:
    risk=c.get('risk_kind')
    trusted=(['A'] if risk in ['parser_failure_scope_reset','catalog_failure_scope_reset','semantic_card_scope']
             else c['plan']['card_keys'] or None)
    if trusted and not set(trusted)<=set(FIX['catalog']): trusted=['A']
    question=semantic_questions.get(risk,'')
    got=P.route(question,c['plan'],FIX['facts'],FIX['catalog'],trusted_scope=trusted,
                parser_failed=c['parser_failed'])
    controlled.append(dict(case=c['name'],risk=risk,question=question,trusted_scope=trusted,result=got))
write('controlled_results.json', {'records':controlled,'synthetic_cases_not_frequency_estimate':True})
semantic=[r for r in controlled if str(r['risk']).startswith('semantic_')]
print('필수 회귀:',sum(r['passed'] for r in green),'/',len(green))
print('고정 의미 반례의 정책 복귀(의미 검사와 범위 검사 합계):',sum(r['result']['route']=='unstructured' for r in semantic),'/',len(semantic))
# 정상 문구의 거절률은 별도 실제 질문에서 측정합니다. 6개 탐지가 전체 의미 검증은 아닙니다.
assert len(semantic)==6 and all(r['result']['route']=='unstructured' for r in semantic)

assert sum(r['result']['reason']=='semantic_conflict' for r in semantic)==5
assert sum(r['result']['reason']=='untrusted_parser_scope' for r in semantic)==1


필수 회귀: 9 / 9
고정 의미 반례의 정책 복귀(의미 검사와 범위 검사 합계): 6 / 6


In [4]:
CATALOG=json.loads((ROOT/'notebooks/data/42_expanded_query_evaluation/catalog.json').read_text())['catalog']
INSTRUCTIONS=load('boundary46_prompt','33_query_routing.py').INSTRUCTIONS+'\n카드 catalog:\n'+F.canonical(CATALOG)
# 실행 입력과 평가 주석을 분리합니다. 평가 값은 API 지시문에 넣지 않습니다.
pairs=[
 ('typo','카페에서 마시는 커피를 10% 이상 할인받는 카드','카페에서 마시는 카피를 10% 이상 할인받는 카드',
  dict(target='커피',action='discount',rate=10)),
 ('typo','넷플릭스 구독료를 10% 이상 할인받는 카드','넷플리스 구독료를 10% 이상 할인받는 카드',
  dict(target='넷플릭스',action='discount',rate=10)),
 ('typo','스타벅스에서 5% 이상 할인되는 카드','스타벅쓰에서 5% 이상 할인되는 카드',
  dict(target='스타벅스',action='discount',rate=5)),
 ('typo','디즈니플러스 구독료를 5% 이상 할인받는 카드','디즈니플라스 구독료를 5% 이상 할인받는 카드',
  dict(target='디즈니플러스',action='discount',rate=5)),
 ('spacing','전월 실적 기준 30만원 이하이고 커피를 10% 이상 할인하는 카드',
  '전월실적기준30만원이하고 커피10%이상할인하는거',dict(target='커피',action='discount',rate=10,spend=300000)),
 ('spacing','GS25와 넷플릭스에서 각각 10% 이상 할인되는 카드 한 장',
  'GS25랑 넷플릭스 둘다 10%이상 할인되는 카드 한장',
  dict(targets=['GS25','넷플릭스'],action='discount',rate=10,boolean='AND')),
 ('spacing','커피 10% 이상 할인에 월 할인한도 5000원 이상인 카드',
  '커피10%이상할인에 월할인한도5천원이상인거',dict(target='커피',action='discount',rate=10,cap=5000,period='month')),
 ('spacing','NH 올원더풀의 백화점 할인 조건을 알려줘',
  'NH올원더풀 백화점할인 조건좀',dict(target='백화점',action='discount',rate=None,scope=['NH/NH_AllWonderful'])),
 ('ambiguous','커피를 10% 이상 할인받는 카드','카피 비용이 10% 이상 할인되는 카드',
  dict(target='커피',action='discount',rate=10)),
 ('ambiguous','네이버페이 결제 혜택이 있는 카드','페이 결제 혜택이 있는 카드',
  dict(target='네이버페이',action='any',rate=None)),
 ('ambiguous','트래블로그 신용카드의 여행 혜택을 알려줘','여행할 때 쓸만한 카드가 뭐야',
  dict(target='여행',action='any',rate=None)),
 ('ambiguous','하나 스카이패스 아멕스 플래티늄 카드의 적립 혜택','스카이패스 카드의 적립 혜택',
  dict(target=None,action='earn',rate=None,scope=['hana/Hana_SKYPASS_Amex_Platinum'])),
]
queries=[]; annotations=[]
for i,(kind,clean,variant,expected) in enumerate(pairs,1):
    for version,text in [('clean',clean),('variant',variant)]:
        qid=f'B{i:02d}_{version}'
        queries.append(dict(query_id=qid,question=text,pair=i,kind=kind,version=version))
        annotation=dict(query_id=qid,expected=expected,requires_clarification=(kind=='ambiguous' and version=='variant'))
        if kind=='ambiguous' and version=='variant':
            annotation['interpretation_note']={9:'복사 비용과 음료 커피의 뜻이 갈림',10:'결제 서비스 제공자를 지정하지 않음',
              11:'일반 여행 범주와 비구조화 모두 허용, 임의 수치/예산 추가 금지',12:'여러 스카이패스 상품 중 대상 불명확'}[i]
            if i==11: annotation['requires_clarification']=False
        annotations.append(annotation)
write('questions.json',{'queries':queries,'split':'development','frozen_before_api':True})
write('evaluation_annotations.json',{'annotations':annotations,'never_sent_to_provider':True})
# 매번 새 반복 키를 쓰되 중단 재개는 같은 키의 저장 응답을 읽습니다. 자동 재시도 없음.
sources=['46_preproduction_common.py','34_followup_common.py','35_answer_experiment.py',
         '36_cost_accounting.py','32_search_experiment.py','30_build_structured_db.py','33_query_routing.py']
manifest=dict(experiment=46,revision='pre_api_review_v2',status='authorized',date='2026-10-21',split='development',
  question_ids=[q['query_id'] for q in queries],questions_sha256=F.digest(queries),
  evaluation_annotations_sha256=F.digest(annotations),repeats=3,logical_parser_conditions=72,
  maximum_physical_requests=72,maximum_retries=0,maximum_embedding_requests=0,
  maximum_retrieval_calls=0,maximum_rerank_calls=0,maximum_answer_requests=0,
  global_cost_cap_usd='8',stage_cost_cap_usd='0.50',latency_rule='실측; 운영 SLA 미정',
  model='gpt-6-luna',reasoning='medium',max_output_tokens=3000,
  resume_rule='동일 반복 키의 저장 응답만 재사용; started/비용 미상 시 자동 재호출 금지',
  allowed_identities={'parser':[f"{q['query_id']}:repeat{r}" for r in range(1,4) for q in queries]},
  allowed_request_by_identity={f"{q['query_id']}:repeat{r}":F.digest([dict(role='user',content=q['question'])]) for r in range(1,4) for q in queries},
  parser_schema_sha256=F.digest(P.E.QueryPlan.model_json_schema()),
  parser_instructions_sha256=F.digest(INSTRUCTIONS),
  source_sha256={f'notebooks/scripts/{s}':hashlib.sha256((ROOT/'notebooks/scripts'/s).read_bytes()).hexdigest() for s in sources},
  data_sha256={str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in [
      ROOT/'notebooks/data/30_structured_benefit_db/facts.jsonl',
      ROOT/'notebooks/data/42_expanded_query_evaluation/catalog.json',
      ROOT/'notebooks/data/43_offline_validation/routing_fixtures.json']},
  mandatory_gate_pass=all(r['passed'] for r in green),
  no_operating_changes=True,new_document_embeddings=0)
if (PATH/'execution_manifest.json').exists():
    previous=F.read_json(PATH/'execution_manifest.json')
    assert previous==manifest, '실행 계약 변경은 별도 버전 없이 허용하지 않습니다.'
else: write('execution_manifest.json',manifest)
print('봉인:',len(queries),'개 질문 × 3회, 전체 8달러 / 이 단계 0.50달러')


봉인: 24 개 질문 × 3회, 전체 8달러 / 이 단계 0.50달러


In [5]:
# 폴더 정리 이후의 실제 파일을 새 기준으로 보존합니다. 과거 봉인 값을 고치지 않습니다.
protection_path=PATH/'protected_files.json'
if not protection_path.exists():
    import subprocess
    tracked=subprocess.check_output(['git','ls-files','-z'],cwd=ROOT).decode().split('\0')
    protected={ROOT/name for name in tracked if name and (ROOT/name).is_file()}
    for base in ['docs','notebooks','data','services','packages','jobs']:
        for p in (ROOT/base).rglob('*'):
            if not p.is_file() or p.is_symlink():continue
            relative=str(p.relative_to(ROOT))
            if any(part in p.parts for part in ['__pycache__','node_modules','.next','execution_cache']):continue
            if relative.startswith(('notebooks/data/46_','notebooks/data/47_','notebooks/data/48_','notebooks/data/49_',
                                    'notebooks/data/00_notebook_layout_migration/')):continue
            if relative.startswith(('notebooks/46_','notebooks/47_','notebooks/48_','notebooks/49_',
                                    'notebooks/scripts/46_','notebooks/scripts/47_','notebooks/scripts/48_','notebooks/scripts/49_',
                                    'docs/report/46_','docs/report/47_','docs/report/48_','docs/report/49_')):continue
            if relative=='notebooks/notes/46_49_preproduction_test_plan.md':continue
            protected.add(p)
    protected={p for p in protected if str(p.relative_to(ROOT))!='notebooks/notes/46_49_preproduction_test_plan.md'}
    write('protected_files.json',{'frozen_after_folder_migration':True,
        'files':[F.file_record(p) for p in sorted(protected)],'excluded_execution_caches':True})
print('보존 대상 파일:',len(F.read_json(protection_path)['files']))


보존 대상 파일: 25289


In [6]:
# 실제 경계의 중단 동작을 검증합니다. 외부 API는 호출하지 않습니다.
original_ledger=P.ledger
api=P.BudgetCache(46,'parser',manifest['allowed_identities']['parser'][0],allow_network=True)
kwargs=dict(instructions=INSTRUCTIONS,input=[dict(role='user',content=queries[0]['question'])],
            text_format=P.E.QueryPlan,tools=[],store=False)
checks=[]
try:
    for label, state, expected in [
        ('global_cap',dict(unknown_cost_records=0,known_cost_usd='7.9999'),'Global 8 USD'),
        ('unknown_usage',dict(unknown_cost_records=1,known_cost_usd='0'),'Unknown prior request cost')]:
        P.ledger=lambda: state
        try:api.reserve(api.normalized(kwargs))
        except RuntimeError as error:
            assert expected in str(error),str(error); checks.append(dict(case=label,passed=True,error=str(error)))
        else:raise AssertionError('유료 호출 전 중단되지 않았습니다: '+label)
finally:P.ledger=original_ledger
write('cost_guard_tests.json',{'records':checks,'new_api_requests':0})
print('비용 경계 검사',len(checks),'통과')

# 잘못된 질문/반복 조합과 저장 행을 실제 검사 함수가 거절해야 합니다.
wrong={**kwargs,'input':[dict(role='user',content=queries[-1]['question'])]}
try:api.reserve(api.normalized(wrong))
except RuntimeError as error:assert 'outside frozen questions' in str(error)
else:raise AssertionError('identity와 다른 질문이 허용됐습니다.')
try:P.validate_parser_row(dict(query_id=queries[-1]['query_id']),identity=api.identity,
    query_id=queries[0]['query_id'],repeat=1,question=queries[0]['question'],
    manifest=manifest,request_key=api.key(kwargs),cache=api.cache)
except RuntimeError as error:assert 'identity/input/manifest mismatch' in str(error)
else:raise AssertionError('다른 질문의 저장 행이 허용됐습니다.')
print('질문/반복 및 저장 행 혼합 거절 2건 통과')


비용 경계 검사 2 통과
질문/반복 및 저장 행 혼합 거절 2건 통과


In [7]:
# 기본 실행은 오프라인입니다. 승인된 유료 실행에서만 환경 값을 1로 지정합니다.
ALLOW_NETWORK=os.getenv('PREPROD46_ALLOW_NETWORK')=='1'
facts=[json.loads(line) for line in (ROOT/'notebooks/data/30_structured_benefit_db/facts.jsonl').read_text().splitlines()]
rows=[]
started=time.perf_counter(); cpu_started=time.process_time()
for repeat in range(1,4):
    for q in queries:
        identity=f"{q['query_id']}:repeat{repeat}"
        destination=PATH/'parser_rows'/f'{identity}.json'
        api=P.BudgetCache(46,'parser',identity,allow_network=ALLOW_NETWORK)
        kwargs=dict(model='gpt-6-luna',instructions=INSTRUCTIONS,
                input=[{'role':'user','content':q['question']}],text_format=P.E.QueryPlan,
                tools=[],store=False,max_output_tokens=3000)
        request_key=api.key(kwargs)
        if destination.exists():
            row=F.read_json(destination)
            P.validate_parser_row(row,identity=identity,query_id=q['query_id'],repeat=repeat,
                question=q['question'],manifest=manifest,request_key=request_key,cache=api.cache)
            rows.append(row);continue
        t=time.perf_counter()
        try:
            response=api.parse(model='gpt-6-luna',instructions=INSTRUCTIONS,
                input=[{'role':'user','content':q['question']}],text_format=P.E.QueryPlan,
                tools=[],store=False,max_output_tokens=3000)
            value=response.output_parsed.model_dump(mode='json') if response.output_parsed else None
            status='ok' if value is not None else 'failed'
            error=None
        except Exception as exc:
            value=None; status='failed';error=f'{type(exc).__name__}: {exc}'
            # 오프라인 캐시 누락이나 비용/봉인 실패를 새 모델 실패로 저장하지 않습니다.
            if not api.events:raise
        trusted=P.explicit_scope(q['question'],CATALOG)
        # 모호한 별칭은 신뢰 가능한 명시 범위를 임의로 만들지 않습니다.
        unclear=('스카이패스카드' in P.compact(q['question']) and trusted is None)
        routing=P.route(q['question'],value,facts,CATALOG,trusted_scope=trusted,
                        parser_failed=status!='ok',scope_unclear=unclear)
        row=dict(identity=identity,request_key=request_key,manifest_sha256=F.digest(manifest),
            question_sha256=F.digest(q['question']),query_id=q['query_id'],repeat=repeat,status=status,error=error,question=q['question'],
            plan=value,routing=routing,trusted_scope=trusted,events=api.events,
            wall_seconds=time.perf_counter()-t,questions_sha256=manifest['questions_sha256'])
        P.validate_parser_row(row,identity=identity,query_id=q['query_id'],repeat=repeat,
            question=q['question'],manifest=manifest,request_key=request_key,cache=api.cache)
        write(f'parser_rows/{identity}.json',row);rows.append(row)
        write('progress.json',{'completed':len(rows),'expected':72,'last_identity':identity,
            'ledger':P.ledger()})
        print(identity,status,routing['reason'],flush=True)
        current=P.ledger()
        if current['unknown_cost_records']:raise RuntimeError('사용량 비용 미상: 추가 유료 요청 중단')
assert len(rows)==72 and len({(r['query_id'],r['repeat']) for r in rows})==72
assert {(r['query_id'],r['repeat']) for r in rows}=={(q['query_id'],r) for q in queries for r in range(1,4)}
current_resources=dict(wall_seconds=time.perf_counter()-started,
    cpu_seconds=time.process_time()-cpu_started,peak_rss_kib=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
    api_ledger=P.ledger(),cost_basis='저장된 API usage × 공식 단가; 실제 청구서 비용은 미측정',
    gpu='미사용',new_document_embeddings=0,new_query_embeddings=0)
previous_result=F.read_json(PATH/'parser_results.json') if (PATH/'parser_results.json').exists() else None
sessions=F.read_json(PATH/'execution_sessions.json') if (PATH/'execution_sessions.json').exists() else {'sessions':[]}
sessions['sessions'].append(dict(allow_network=ALLOW_NETWORK,completed_rows=len(rows),resources=current_resources))
write('execution_sessions.json',sessions)
write('parser_results.json',{'records':rows,'resources':previous_result['resources'] if previous_result else current_resources})
print('완료',len(rows),P.ledger())


완료 72 {'record_count': 72, 'known_priced_records': 72, 'unknown_cost_records': 0, 'duplicate_saved_responses': 0, 'known_cost_usd': '0.009009430', 'tokens': {'input_tokens': 210063, 'output_tokens': 13005, 'ordinary_input_tokens': 216, 'cached_input_tokens': 206483, 'cache_write_tokens': 3364, 'embedding_tokens': 0}, 'billed_cost_usd': None}


In [8]:
import statistics
by_id={a['query_id']:a for a in annotations}
metadata={q['query_id']:q for q in queries}
aliases={
 '커피':{'커피','카페','스타벅스','투썸','이디야','폴바셋','커피빈'},
 '스타벅스':{'스타벅스'},'넷플릭스':{'넷플릭스'},'디즈니플러스':{'디즈니플러스'},
 '여행':{'여행','항공','호텔','숙박','공항','라운지','면세점'}}
def score(row):
    annotation=by_id[row['query_id']]; expected=annotation['expected']
    value=row['plan']; q=metadata[row['query_id']]
    if not value:return dict(parser_success=False,semantic_match=False,clarification=False,eligible_loss=None)
    reqs=[r for group in value['groups'] for r in group]
    clarification=bool(value['unresolved']) or row['routing']['route']=='clarify'
    if annotation['requires_clarification']:
        matched=clarification
    elif q['pair']==11 and q['version']=='variant':
        # 포괄 여행 조언은 여행 범주 또는 비구조화 허용. 임의 수치·명시 카드 제한 금지.
        matched=(not value['card_keys'] and all(r[k] is None for r in reqs
                    for k in ['min_percent','max_spend_krw','min_cap_krw']))
    else:
        targets=expected.get('targets') or ([expected['target']] if expected.get('target') else [])
        matched=True
        for target in targets:
            family=aliases.get(target,{target})
            candidates=[r for r in reqs if set(r['targets']) & family]
            matched &= any((expected['action']=='any' or r['action']==expected['action']) and
              r['min_percent']==expected['rate'] and r['max_spend_krw']==expected.get('spend') and
              r['min_cap_krw']==expected.get('cap') and r['cap_period']==expected.get('period','any')
              for r in candidates)
        if not targets:
            matched &= all(r['action']==expected['action'] and r['min_percent'] is None for r in reqs)
        matched &= set(value['card_keys'])==set(expected.get('scope',[]))
        if expected.get('boolean')=='AND':
            matched &= len(value['groups'])==1 and len(value['groups'][0])>=2
    return dict(parser_success=True,semantic_match=bool(matched),clarification=clarification,
                structured_routed=row['routing']['route']=='structured_then_unstructured',
                guard_semantic_rejection=row['routing']['reason']=='semantic_conflict',
                explicit_scope_preserved=(row['trusted_scope'] is None or row['routing']['scope'] is not None and
                   set(row['routing']['scope'])<=set(row['trusted_scope'])))
evaluated=[dict(**metadata[r['query_id']],repeat=r['repeat'],
                score=score(r),route=r['routing']['route'],reason=r['routing']['reason']) for r in rows]
groups=[]
for kind in ['typo','spacing','ambiguous']:
    for version in ['clean','variant']:
        subset=[r for r in evaluated if r['kind']==kind and r['version']==version]
        groups.append(dict(kind=kind,version=version,n=len(subset),
            parser_success=sum(r['score']['parser_success'] for r in subset),
            semantic_match=sum(r['score']['semantic_match'] for r in subset),
            clarify=sum(r['score']['clarification'] for r in subset),
            guarded=sum(r['route']=='unstructured' and r['reason']=='semantic_conflict' for r in subset)))
latencies=[event['wall_seconds'] for r in rows for event in r['events']
           if not event.get('cache_hit') and 'wall_seconds' in event]
latency=dict(n=len(latencies),mean_seconds=statistics.mean(latencies) if latencies else None,
    median_seconds=statistics.median(latencies) if latencies else None,
    p95_seconds=sorted(latencies)[max(0,__import__('math').ceil(.95*len(latencies))-1)] if latencies else None,
    max_seconds=max(latencies) if latencies else None)
summary=dict(records=evaluated,groups=groups,latency=latency,api_ledger=P.ledger(),
  mandatory_regression_pass=all(r['passed'] for r in green),
  semantic_literal_flags_detected=sum(r['result']['reason']=='semantic_conflict' for r in semantic),
  semantic_scope_policy_rejected=sum(r['result']['reason']=='untrusted_parser_scope' for r in semantic),
  semantic_controlled_fallback=sum(r['result']['route']=='unstructured' for r in semantic),
  semantic_controlled_total=len(semantic),operating_code_modified=False,
  limitations=['개발 질문이며 독립 확인 평가 아님','6개 의미 반례 통과는 임의 의미 오류 해결 보장 아님',
      '정답 카드 회수·최종 답변 품질은 이 parser 전용 단계에서 미측정','실제 청구서 금액·운영 SLA 미측정'])
write('evaluation_results.json',summary)
print(json.dumps(groups,ensure_ascii=False,indent=2))
print('응답시간:',latency,'비용:',summary['api_ledger'])
# 현재 코드를 읽기 전용으로 썼는지 보존 계약과 대조합니다.
changed=[]; allowed_document_appends=[]
allowance=F.read_json(PATH/'authorized_metric_append.json') if (PATH/'authorized_metric_append.json').exists() else None
for record in F.read_json(protection_path)['files']:
    source=ROOT/record['path']
    if not source.exists():changed.append(record['path']);continue
    if F.file_record(source)!=record:
        body=source.read_bytes()
        approved=(allowance and record['path']==allowance['path'] and
            hashlib.sha256(body).hexdigest()==allowance['current_sha256'] and
            hashlib.sha256(body[:record['size']]).hexdigest()==record['sha256'])
        if approved:allowed_document_appends.append(record['path'])
        else:changed.append(record['path'])
write('preservation_verification.json',dict(checked=len(F.read_json(protection_path)['files']),
    changed=changed,allowed_document_appends=allowed_document_appends,passed=not changed))
assert not changed,changed


[
  {
    "kind": "typo",
    "version": "clean",
    "n": 12,
    "parser_success": 12,
    "semantic_match": 12,
    "clarify": 0,
    "guarded": 0
  },
  {
    "kind": "typo",
    "version": "variant",
    "n": 12,
    "parser_success": 12,
    "semantic_match": 11,
    "clarify": 0,
    "guarded": 0
  },
  {
    "kind": "spacing",
    "version": "clean",
    "n": 12,
    "parser_success": 12,
    "semantic_match": 12,
    "clarify": 0,
    "guarded": 0
  },
  {
    "kind": "spacing",
    "version": "variant",
    "n": 12,
    "parser_success": 12,
    "semantic_match": 11,
    "clarify": 0,
    "guarded": 1
  },
  {
    "kind": "ambiguous",
    "version": "clean",
    "n": 12,
    "parser_success": 12,
    "semantic_match": 9,
    "clarify": 0,
    "guarded": 0
  },
  {
    "kind": "ambiguous",
    "version": "variant",
    "n": 12,
    "parser_success": 12,
    "semantic_match": 7,
    "clarify": 4,
    "guarded": 0
  }
]
응답시간: {'n': 72, 'mean_seconds': 3.268663533388715, 'median_

In [9]:
rows=F.read_json(PATH/'parser_results.json')['records']
facts=[json.loads(line) for line in (ROOT/'notebooks/data/30_structured_benefit_db/facts.jsonl').read_text().splitlines()]
# 유료 결과를 본 뒤 수행하는 개발 진단입니다. 원래 결과·주석·정책은 덮어쓰지 않습니다.
# 대상의 한국어/영문 문자열 차이는 단독 거절 근거로 채택하지 않습니다.
def selected_route(question,value,facts,catalog,**kwargs):
    original=P.route(question,value,facts,catalog,**kwargs)
    flags=original['semantic_flags']
    if original['reason']=='semantic_conflict' and flags and all(f.startswith('missing_literal_target:') for f in flags):
        result=P.route(question,value,facts,catalog,check_semantics=False,**kwargs)
        return {**result,'diagnostic_target_flags':flags}
    return original
selected=[]
for c in REGRESSIONS:
    got=selected_route('',c['value'],FIX['facts'],FIX['catalog'],trusted_scope=c['explicit'],
         parser_failed=c['failed'],scope_unclear=c['unclear'])
    selected.append(dict(id=c['id'],passed=all(got[k]==v for k,v in c['want'].items())))
assert all(r['passed'] for r in selected)
selected_controlled=[]
for c in controlled:
    case=next(case for case in FIX['cases'] if case['name']==c['case'])
    got=selected_route(c['question'],case['plan'],FIX['facts'],FIX['catalog'],
         trusted_scope=c['trusted_scope'],parser_failed=case['parser_failed'])
    selected_controlled.append(dict(case=c['case'],risk=c['risk'],result=got))
known_semantic=[r for r in selected_controlled if str(r['risk']).startswith('semantic_')]
assert sum(r['result']['route']=='unstructured' for r in known_semantic)==5
# 72개 원관측은 유지하고 개발 결과를 본 뒤 선정한 정책을 따로 집계합니다.
policy_rows=[]
for r in rows:
    got=selected_route(r['question'],r['plan'],facts,CATALOG,trusted_scope=r['trusted_scope'],
         parser_failed=r['status']!='ok',scope_unclear=r['routing']['reason']=='unclear_explicit_scope')
    policy_rows.append(dict(query_id=r['query_id'],repeat=r['repeat'],original=r['routing'],selected=got))
# 기존48질문은 저장 파서를 로컬에서 재생한 개발 진단입니다.
prior=ROOT/'notebooks/data/42_expanded_query_evaluation'
prior_questions=json.loads((prior/'questions.json').read_text())['queries']
prior_plans={r['query_id']:r for r in json.loads((prior/'parser_results.json').read_text())['records']}
actual=[]
for q in prior_questions:
    saved=prior_plans[q['query_id']]
    trusted=P.explicit_scope(q['question'],CATALOG)
    result=selected_route(q['question'],saved['plan'],facts,CATALOG,trusted_scope=trusted,
         parser_failed=saved['status']!='ok')
    actual.append(dict(query_id=q['query_id'],question=q['question'],result=result))
write('selected_policy.json',dict(version='46-boundary-selected-v1',development_selected_after_results=True,
  mandatory_gate_pass=all(r['passed'] for r in selected),
  rules=['schema_empty_failure_and_trusted_scope_guard','explicit_percent_action_period_boolean_conflict_fallback',
         'literal_target_flags_diagnostic_only','ambiguous_explicit_SKYPASS_scope_clarify'],
  hard_target_literal_check=False,kept_original_question=True,operating_adopted=False,
  code_reference='46_query_boundary_validation.ipynb:policy_audit',shared_helper_sha256=F.digest(
    (ROOT/'notebooks/scripts/46_preproduction_common.py').read_bytes()),
  known_controlled_semantic_fallbacks=5,known_controlled_semantic_cases=6,
  unresolved_risk='카드 범위가 남는 잘못된 대상 의미 추출은 완전히 탐지하지 못함;49번 확인 전 운영 이식 적격 아님'))
write('post_run_policy_audit.json',dict(regressions=selected,controlled=selected_controlled,actual_parser_rows=policy_rows,
    previous48_offline_diagnostics=actual,new_api_requests=0,
    annotation_issue=dict(query_id='B11_clean',repeats=[1,2,3],kind='frozen_gold_scope_omission',
      source_question='트래블로그 신용카드의 여행 혜택을 알려줘',
      missing_expected_scope=['hana/Hana_Travellog'],action='원래 자동점수는 보존; 올바른 명시 카드 추출3건을 모델 오류로 해석하지 않음'),
    target_normalization_issues=[dict(query_id='B02_variant',repeat=2,output='넷플릭스 구독료',
      consequence='의미상 대상은 맞지만 DB 대상 정규화와 안 맞아 후보0; 비구조화 복귀'),
      dict(query_id='B06_variant',repeat=3,output='Netflix',
      consequence='영문 별칭을 문자 검사가 오거절. 단독 문자열 거절 규칙 미선정; DB 영문 별칭 문제는 잔여')]))
print('선정 경계 필수9건 통과; 고정 의미 반례 복귀5/6, 대상 의미 오류1건 잔여')
print('기존48개 저장 계획은 새 유료 호출 없이 현재 정책으로 진단 완료')


선정 경계 필수9건 통과; 고정 의미 반례 복귀5/6, 대상 의미 오류1건 잔여
기존48개 저장 계획은 새 유료 호출 없이 현재 정책으로 진단 완료
